In [45]:
from selenium import webdriver 
from selenium.webdriver.common.by import By 
from selenium.webdriver.support.ui import WebDriverWait 
from selenium.webdriver.support import expected_conditions as EC 
from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementClickInterceptedException


In [46]:
import pandas as pd

url = "https://www.flashscore.pt/futebol/portugal/liga-portugal-betclic/resultados/"

driver = webdriver.Chrome()
driver.get(url)

# Close the cookie consent banner
try:
    accept_cookies = WebDriverWait(driver, 5).until(
        EC.presence_of_element_located((By.ID, "onetrust-accept-btn-handler"))
    )
    driver.execute_script("arguments[0].click();", accept_cookies)
except (TimeoutException, NoSuchElementException):
    pass

while True:
    try:
        more_results = WebDriverWait(driver, 5).until(
            EC.element_to_be_clickable((By.XPATH, "//*[@id='tournamentPage']/div[2]/section/div[2]/button"))
        )
        driver.execute_script("arguments[0].click();", more_results)
    except (TimeoutException, ElementClickInterceptedException):
        break


elements = driver.find_elements(By.XPATH, "//*[starts-with(@id, 'match-row-g_')]")
hrefs = [el.get_attribute('href') for el in elements]


data_df = pd.DataFrame({'href': hrefs})
display(data_df)

,href
0,https://www.flashscore.pt/jogo/futebol/casa-pi...
1,https://www.flashscore.pt/jogo/futebol/casa-pi...
2,https://www.flashscore.pt/jogo/futebol/benfica...
3,https://www.flashscore.pt/jogo/futebol/alverca...
4,https://www.flashscore.pt/jogo/futebol/gil-vic...
...,...
303,https://www.flashscore.pt/jogo/futebol/alverca...
304,https://www.flashscore.pt/jogo/futebol/famalic...
305,https://www.flashscore.pt/jogo/futebol/afs-IZP...
306,https://www.flashscore.pt/jogo/futebol/gil-vic...


In [ ]:
from urllib.parse import urlparse, parse_qs
from selenium.common.exceptions import TimeoutException, NoSuchElementException

COLUMNS = ['player', 'position', 'rating', 'total_remates', 'xG',
           'passes_certos', 'toques', 'toques_area_adv', 'dribles', 'duelos']

def get_match_info(driver):
    try:
        container = driver.find_element(By.XPATH, "//*[@id='detail']/div[2]")
        home = container.find_element(By.CSS_SELECTOR, ".duelParticipant__home .participant__participantName a").text
        away = container.find_element(By.CSS_SELECTOR, ".duelParticipant__away .participant__participantName a").text
        spans = container.find_elements(By.CSS_SELECTOR, ".detailScore__wrapper span")
        score = f"{spans[0].text}-{spans[2].text}" if len(spans) >= 3 else None
        return home, away, score
    except NoSuchElementException:
        return None, None, None

all_dfs = []

for href in data_df['href'].head(3):
    if not href:
        continue

    mid = parse_qs(urlparse(href).query).get('mid', [None])[0]
    if not mid:
        continue

    new_url = href.replace('/?mid=', '/sumario/estatisticas-jogadores/principais/?mid=')
    driver.get(new_url)

    try:
        table_el = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, f"//*[@id='widget-player-match-stats-{mid}']/div[2]/div[1]/table"))
        )

        home, away, score = get_match_info(driver)

        rows_data = []
        for row in table_el.find_elements(By.CSS_SELECTOR, "tbody tr"):
            cells = row.find_elements(By.CSS_SELECTOR, "td")
            if not cells:
                continue
            try:
                name = cells[0].find_element(By.CSS_SELECTOR, ".fp-playerName_E6lgN").text
            except NoSuchElementException:
                name = ''
            try:
                position = cells[0].find_element(By.CSS_SELECTOR, "[data-testid='wcl-scores-caption-05']").text
            except NoSuchElementException:
                position = ''
            other = [c.text for c in cells[1:]]
            rows_data.append([name, position] + other)

        if rows_data:
            n_cols = len(rows_data[0])
            df = pd.DataFrame(rows_data, columns=COLUMNS[:n_cols])
            df['match_id'] = mid
            df['home'] = home
            df['away'] = away
            df['score'] = score
            all_dfs.append(df)

    except TimeoutException:
        pass

general_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()
display(general_df)



In [48]:
general_df.to_csv('player_stats.csv', index=False)
print(f"Guardado: {len(general_df)} linhas em player_stats.csv")

Guardado: 9662 linhas em player_stats.csv


In [49]:
display(general_df)


,player,position,rating,total_remates,xG,passes_certos,toques,toques_area_adv,dribles,duelos,match_id,home,away,score
0,Larrazabal G.,Médio ofensivo,8.3,1,0.31,8/17 (47%),46,3,0/2 (0%),5,IcJBYh6e,Casa Pia AC,Torreense,2-0
1,Kaly,Defesa-central,8.1,1,0.21,17/18 (94%),38,-,1/1 (100%),11,IcJBYh6e,Casa Pia AC,Torreense,2-0
2,Ofori L.,Médio,7.8,1,0.91,16/18 (89%),29,2,-,5,IcJBYh6e,Casa Pia AC,Torreense,2-0
3,Prieto K.,Ala,7.8,2,0.03,13/18 (72%),45,1,1/2 (50%),14,IcJBYh6e,Casa Pia AC,Torreense,2-0
4,Geraldes A.,Ala,7.6,-,-,23/28 (82%),56,-,2/3 (67%),7,IcJBYh6e,Casa Pia AC,Torreense,2-0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9657,Nsona K.,Médio,5.2,-,-,7/12 (58%),25,3,1/1 (100%),8,YDjUoANH,Casa Pia AC,Sporting CP,0-2
9658,Harder C.,Avançado,-,1,0.06,3/4 (75%),6,2,-,-,YDjUoANH,Casa Pia AC,Sporting CP,0-2
9659,Kochorashvili G.,Médio,-,1,0.26,16/16 (100%),20,2,-,-,YDjUoANH,Casa Pia AC,Sporting CP,0-2
9660,Miguel Sousa,Médio,-,-,-,6/6 (100%),8,-,-,1,YDjUoANH,Casa Pia AC,Sporting CP,0-2
